In [1]:
device_rules_simple = [
    {"name": "mobile", "score": 5},
    {"name": "pc",     "score": 10},
]

def device_risk_by_table(device):
    risk = 0
    for rule in device_rules_simple:
        if device == rule["name"]:
            risk += rule["score"]
    return risk

def exercise_device():
    devices = ["mobile", "pc", "mobile", "pc"]
    for d in devices:
        r = device_risk_by_table(d)
        print(f"device={d}, risk={r}")

exercise_device()


device=mobile, risk=5
device=pc, risk=10
device=mobile, risk=5
device=pc, risk=10


In [2]:
# 登录数据按城市统计
logins = [
    {"user_id": 101, "device": "mobile", "city": "Beijing",  "fail_count": 0},
    {"user_id": 102, "device": "pc",     "city": "Shanghai", "fail_count": 1},
    {"user_id": 103, "device": "mobile", "city": "Other",    "fail_count": 3},
    {"user_id": 104, "device": "pc",     "city": "Other",    "fail_count": 5},
    {"user_id": 105, "device": "mobile", "city": "Beijing",  "fail_count": 2},
]

city_stats = {}
for tx in logins:
    city = tx["city"]
    fail = tx["fail_count"]
    if city not in city_stats:
        city_stats[city] = {"count":0,"fail":0}
    city_stats[city]["count"] += 1
    city_stats[city]["fail"] += fail
print(city_stats)

for city,stats in city_stats.items():
    count = stats["count"]
    fail = stats["fail"]
    avg_fail = fail / count
    print(f"城市{city}:登录次数={count},平均输错次数={avg_fail:.2f}")



{'Beijing': {'count': 2, 'fail': 2}, 'Shanghai': {'count': 1, 'fail': 1}, 'Other': {'count': 2, 'fail': 8}}
城市Beijing:登录次数=2,平均输错次数=1.00
城市Shanghai:登录次数=1,平均输错次数=1.00
城市Other:登录次数=2,平均输错次数=4.00


## 学习计划（数据分析 / 风控策略方向）

**你目前已掌握：**
- 基础：列表、字典、for 循环、if/elif、f-string
- 函数：单维规则（金额/渠道/设备/城市/新用户等）+ 组合风险分
- 字典列表：`[{"id":1, "amount":100}, ...]` 的遍历与取值
- 规则表驱动：用列表存规则，循环匹配（如 device_rules_simple）
- 按维度聚合：用字典做分组统计（如 city_stats：count、fail、平均）
- 条件筛选：按 risk >= 某阈值只处理部分数据
- 简单统计：累加、求平均、找最大值及对应 id

**建议接下来按顺序练：**
1. **排序与 Top N**：用 `sorted(列表, key=...)` 按风险分排序，输出前 3 笔高风险（实习里常做「风险从高到低排，先看最差的」）
2. **封装「筛选+统计」**：写一个函数，输入「交易列表 + 风险阈值」，返回「高风险笔数」或「高风险列表」（为后面写可复用的分析函数打基础）
3. **列表推导式**：用 `[x for x in list if 条件]` 替代 for+append，代码更短
4. **pandas 入门**：用 `pd.DataFrame(列表)` 转成表，练 `df.groupby().agg()`、`df[df["risk"]>=40]`，和现在纯 Python 做对比

**小提醒：** `dic_def.ipynb` 里 exercise2 有一处笔误：`newuser_risk["is_new_user"]` 应改为**调用函数** `newuser_risk(tx["is_new_user"])`，且那一行应是 `risk += ...` 而不是 `is_new_user += ...`，修好后多规则交易风险就能跑通。

### 下一个练习：按风险分排序，输出 Top 3 高风险登录

**要求：**
1. 使用上面已有的 `logins` 和按设备打分的逻辑（或你写的 `device_risk_by_table`），先给每条登录算一个 `risk`。
2. 按 `risk` **从高到低**排序（提示：`sorted(列表, key=函数, reverse=True)`，其中 key 可以是一个 lambda 或函数，接收一条记录返回该记录的 risk）。
3. 取前 3 条，打印：`user_id, device, city, risk`。

**扩展（可选）：** 写一个函数 `top_risk_logins(login_list, n=3)`，参数为登录列表和要取的前 n 条，返回「按风险从高到低排序后的前 n 条」。

In [3]:
# 练习：Top 3 高风险登录 — 完整示例（可直接运行理解）

logins = [
    {"user_id": 101, "device": "mobile", "city": "Beijing",  "fail_count": 0},
    {"user_id": 102, "device": "pc",     "city": "Shanghai", "fail_count": 1},
    {"user_id": 103, "device": "mobile", "city": "Other",    "fail_count": 3},
    {"user_id": 104, "device": "pc",     "city": "Other",    "fail_count": 5},
    {"user_id": 105, "device": "mobile", "city": "Beijing",  "fail_count": 2},
]

# 你写的三个小规则：接收「一个值」，返回该维度的风险分
def city_risk(city):
    if city == "Beijing":
        return 0
    elif city == "Shanghai":
        return 5
    else:
        return 10

def device_risk(device):
    if device == "mobile":
        return 5
    else:
        return 10   # pc 通常比 mobile 风险高一点

def failcount_risk(fail):
    if fail == 0:
        return 0
    elif 1 <= fail <= 3:
        return 5
    else:
        return 10

# 一条登录的总风险 = 设备 + 城市 + 输错次数
def calc_login_risk(one_login):
    r = 0
    r += device_risk(one_login["device"])
    r += city_risk(one_login["city"])
    r += failcount_risk(one_login["fail_count"])
    return r

# 排序：key 表示「按什么来排」，这里按「这条登录的风险分」从高到低
sorted_logins = sorted(logins, key=calc_login_risk, reverse=True)

# 取前 3 条
top3 = sorted_logins[:3]

print("Top 3 高风险登录：")
for t in top3:
    risk = calc_login_risk(t)
    print(f"  user_id={t['user_id']}, device={t['device']}, city={t['city']}, risk={risk}")

# ---------- 扩展：封装成函数，方便以后复用 ----------
def top_risk_logins(login_list, n=3):
    """按风险从高到低排序，返回前 n 条"""
    sorted_list = sorted(login_list, key=calc_login_risk, reverse=True)
    return sorted_list[:n]

# 用法示例：
# result = top_risk_logins(logins, n=3)
# for t in result: ...


Top 3 高风险登录：
  user_id=104, device=pc, city=Other, risk=30
  user_id=102, device=pc, city=Shanghai, risk=20
  user_id=103, device=mobile, city=Other, risk=20


In [ ]:
credit_transactions = [
    {"tx_id": "T001", "amount": 500,   "hour": 14, "is_cross_border": False},
    {"tx_id": "T002", "amount": 15000, "hour": 2,  "is_cross_border": True},
    {"tx_id": "T003", "amount": 8000,  "hour": 10, "is_cross_border": False},
    {"tx_id": "T004", "amount": 200,   "hour": 3,  "is_cross_border": False},
    {"tx_id": "T005", "amount": 12000, "hour": 19, "is_cross_border": True},
]

def calc_fraud_risk(tx):
    risk = 0
    
    # TODO 
    amount = tx["amount"]
    hour = tx["hour"]
    is_cross_border = tx["is_cross_border"]
    
    # TODO 
    if amount > 10000:
        risk += 30
    elif 1000 <= amount <= 10000:
        risk += 10
        
    # TODO 
    if 0 <= hour <= 5:
        risk += 20
        
    # TODO 
    if is_cross_border is True:
        risk += 15
        
    return risk

# TODO 
sorted_tx = sorted(credit_transactions, key=calc_fraud_risk, reverse=True)

# 4. 提取并打印 Top 2 高危交易
# TODO 6: 用切片取出前 2 个
top2 = sorted_tx[:2]

print("Top 2 高风险盗刷嫌疑交易：")
for t in top2:
    # 这里为了打印，我们让评估员再算一次分
    score = calc_fraud_risk(t) 
    print(f"交易号: {t['tx_id']}, 金额: {t['amount']}, 风险总分: {score}")

Top 2 高风险盗刷嫌疑交易：
交易号: T002, 金额: 15000, 风险总分: 65
交易号: T005, 金额: 12000, 风险总分: 45
